# BRCA Metabolic Flux Pipeline — 01. Data processing

Prepares every input needed downstream, from the raw TCGA-BRCA downloads:

1. **Sample inventory** — one row per expression sample (tumour / normal / metastasis, patient, pairing, tissue source site, which data sources cover it).
2. **Transcriptomic files for XomicsToModel** — one `genes,expVal` file per sample (Entrez IDs, linear FPKM). These are the inputs of `modelGen.m`.
3. **Processed clinical table** — one row per sample with harmonised clinical, receptor, subtype and survival variables (read by notebook 02).
4. **Anonymised clinical table** for the supplementary material (no TCGA barcodes, no dates, integer ages).

Nothing is written inside the repository: inputs are read from `Data/rawData` and outputs are written to `results/processedData` in the project Drive folder.

| Output (in `results/processedData/`) | Used by |
|---|---|
| `xomics_files/<sample>_Xomics.txt` | `modelGen.m` (XomicsToModel) |
| `xomics_manifest.csv`, `ensembl_to_entrez.csv` | QC / reproducibility |
| `sample_inventory.csv` | notebook 02, MATLAB |
| `clinical_processed.csv` | notebook 02 |
| `supplementary/S_Table_clinical_anonymized.csv` | supplementary material |
| `run_log_01.json` | reproducibility |

**How to run:** set `DRIVE_DIR` in the configuration cell (or the environment variable `BRCA_DRIVE_DIR`) and run all cells. Requirements: `pandas`, `numpy`, `openpyxl`, and `mygene` (only the first time, to build the Ensembl→Entrez map; afterwards the cached map is used).

## 0. Configuration

In [ ]:
import os, re, json, sys, datetime, platform
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- paths
# Project folder in Google Drive (synced locally). Override with the environment variable BRCA_DRIVE_DIR.
DRIVE_DIR = Path(os.environ.get("BRCA_DRIVE_DIR",
                                Path.home() / "work/Drive/GIMI/01_posgrado/MCBCI_analisisCancer"))

RAW_DIR  = DRIVE_DIR / "Data" / "rawData"
REF_DIR  = DRIVE_DIR / "Data" / "reference"
PROC_DIR = DRIVE_DIR / "results" / "processedData"

EXPRESSION_FILE = RAW_DIR / "TCGA-BRCA_log_FPKM.tsv"       # GDC/Xena, Ensembl IDs x samples
CLINICAL_FILE   = RAW_DIR / "TCGA-BRCA.clinical.tsv"       # GDC clinical, one row per sample
SURVIVAL_FILE   = RAW_DIR / "TCGA-BRCA.survival.tsv.gz"    # UCSC Xena, OS / OS.time per sample
METADATA_FILE   = RAW_DIR / "MetaData.xlsx"                # UCSC Xena phenotype (ER/PR/HER2, PAM50, ancestry)
RECON3D_GENES   = REF_DIR / "Recon3D_genes.xlsx"           # optional: Recon3D gene list (e.g. 3939.1) for QC

XOMICS_DIR   = PROC_DIR / "xomics_files"                   # only *_Xomics.txt here (modelGen.m reads every .txt)
MAP_FILE     = PROC_DIR / "ensembl_to_entrez.csv"          # cached Ensembl -> Entrez map
SUPPL_DIR    = PROC_DIR / "supplementary"

# ---------------------------------------------------------------- parameters
# Scale of EXPRESSION_FILE. The GDC/Xena file is log2(FPKM+1) and was log2-transformed a second time
# before it was saved, i.e. log2(log2(FPKM+1)+1). The scale is verified below on GAPDH/ACTB.
INPUT_SCALE      = "log2log2"        # "log2log2" | "log2" | "linear"
DUPLICATE_RULE   = "max"             # several Ensembl IDs -> one Entrez ID: keep the max (linear FPKM)
REFRESH_MAP      = False             # True: query MyGene.info again even if MAP_FILE exists
WRITE_XOMICS     = True              # False: run all checks without writing the 1,226 files
SAMPLE_LIMIT     = None              # e.g. 10 for a quick test run
XOMICS_THRESHOLD = 0                 # value to use in modelGen.m: param.transcriptomicThreshold
                                     # XomicsToModel marks a gene inactive if expVal < exp(threshold) = 1 FPKM
# >>> CONFIG END

In [ ]:
for p in [EXPRESSION_FILE, CLINICAL_FILE, SURVIVAL_FILE, METADATA_FILE]:
    print(("OK      " if p.exists() else "MISSING ") + str(p.relative_to(DRIVE_DIR)))
PROC_DIR.mkdir(parents=True, exist_ok=True)
SUPPL_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

The expression matrix is read as `float32` (≈300 MB in memory for 60,660 genes × 1,226 samples).

In [ ]:
hdr = pd.read_csv(EXPRESSION_FILE, sep="\t", nrows=0).columns.tolist()
id_col, samples = hdr[0], hdr[1:]
if SAMPLE_LIMIT:
    samples = samples[:SAMPLE_LIMIT]
expr = pd.read_csv(EXPRESSION_FILE, sep="\t", index_col=0, usecols=[id_col] + samples,
                   dtype={s: np.float32 for s in samples})
print(f"Expression: {expr.shape[0]:,} genes x {expr.shape[1]:,} samples")
assert expr.columns.is_unique, "duplicated sample columns"

clin = pd.read_csv(CLINICAL_FILE, sep="\t", low_memory=False)
surv = pd.read_csv(SURVIVAL_FILE, sep="\t")
meta = pd.read_excel(METADATA_FILE)
print(f"Clinical: {clin.shape}   Survival: {surv.shape}   MetaData: {meta.shape}")

## 2. Sample inventory

TCGA barcode `TCGA-TSS-PPPP-SSV`: `TSS` tissue source site, `PPPP` participant, `SS` sample type (01 primary tumour, 06 metastasis, 11 solid tissue normal), `V` vial.
Patients with more than one tumour vial keep all vials (one model each); `is_first_tumor_vial` marks one tumour per patient for analyses that need independent patients.

In [ ]:
TYPE = {"01": "Tumor", "06": "Metastatic", "11": "Normal"}

inv = pd.DataFrame({"sample": expr.columns})
inv["patient"]     = inv["sample"].str[:12]
inv["tss_code"]    = inv["sample"].str[5:7]
inv["type_code"]   = inv["sample"].str[13:15]
inv["vial"]        = inv["sample"].str[15:16]
inv["sample_type"] = inv["type_code"].map(TYPE).fillna("Other")

by_pat = inv.groupby("patient")["sample_type"]
inv["n_samples_patient"] = by_pat.transform("size")
inv["patient_has_tumor"]  = by_pat.transform(lambda s: (s == "Tumor").any())
inv["patient_has_normal"] = by_pat.transform(lambda s: (s == "Normal").any())
inv["paired_tumor_normal"] = inv["patient_has_tumor"] & inv["patient_has_normal"]
first_tumor = (inv[inv.sample_type == "Tumor"].sort_values(["patient", "vial"])
               .drop_duplicates("patient")["sample"])
inv["is_first_tumor_vial"] = inv["sample"].isin(first_tumor)

# coverage by the other data sources
meta_ids = set(meta["hidden"].astype(str).str.replace(".", "-", regex=False).str[:16])
inv["in_clinical"] = inv["sample"].isin(set(clin["sample"]))
inv["in_survival"] = inv["sample"].isin(set(surv["sample"]))
inv["in_metadata"] = inv["sample"].isin(meta_ids) | inv["patient"].isin(set(meta["Patient ID"]))

tss_name = clin.drop_duplicates("submitter_id").set_index(clin.drop_duplicates("submitter_id")["submitter_id"].str[5:7])
tss_name = tss_name["name.tissue_source_site"].to_dict()
inv["tss_name"] = inv["tss_code"].map(tss_name)

print(inv["sample_type"].value_counts().to_string(), "\n")
print(f"patients: {inv.patient.nunique():,}   paired tumour-normal patients: "
      f"{inv[inv.paired_tumor_normal].patient.nunique()}   patients with >1 tumour vial: "
      f"{(inv[inv.sample_type=='Tumor'].groupby('patient').size() > 1).sum()}")
print(inv[["in_clinical", "in_survival", "in_metadata"]].mean().map("{:.1%}".format).to_string())

**Known artefact check.** Samples without MetaData (no ER/PR/HER2, PAM50, ancestry) are not missing at random: in the previous analysis they were concentrated in one tissue source site and formed most of the former "Divergent" group. This table shows where they come from; notebook 02 tests clusters against `in_metadata` and `tss_code`.

In [ ]:
tum = inv[inv.sample_type == "Tumor"]
miss = (tum.assign(no_metadata=~tum.in_metadata)
           .groupby(["tss_code", "tss_name"], dropna=False)["no_metadata"]
           .agg(n_tumors="size", n_without_metadata="sum"))
miss["pct_without_metadata"] = (100 * miss.n_without_metadata / miss.n_tumors).round(1)
print(f"Tumour samples without MetaData: {(~tum.in_metadata).sum()} of {len(tum)}")
miss[miss.n_without_metadata > 0].sort_values("n_without_metadata", ascending=False)

## 3. Expression → `genes,expVal` files for XomicsToModel

Rules (each one fixes a problem of the former `creator.py`):

1. **Genes with expression 0 are kept.** XomicsToModel treats genes missing from the file as NaN (neither active nor inactive), but genes with `expVal < exp(param.transcriptomicThreshold)` as **inactive**, which lets it remove their reactions.
2. **`*_PAR_Y` rows are removed** before aggregation. They are the chrY copies of pseudo-autosomal genes and are always 0 here; averaging them with the chrX copy halved those genes (e.g. CD99, SLC25A6, ASMT, PPP2R3B — the last three are Recon3D genes).
3. **Linear FPKM.** XomicsToModel compares `expVal` with `exp(threshold)` and builds the thermoKernel weights from `log(expVal)`, so it expects a linear scale. The raw file is `log2(log2(FPKM+1)+1)`; the scale is verified with two housekeeping genes (median FPKM of GAPDH and ACTB must be > 100).
4. **Duplicates**: Ensembl IDs mapping to the same Entrez ID are aggregated with the maximum.
5. **Reproducible mapping**: the Ensembl→Entrez map (MyGene.info, best-scoring hit) is saved and reused.

In `modelGen.m` use `param.transcriptomicThreshold = 0` → a gene is active if FPKM ≥ 1.

In [ ]:
def to_linear(v, scale):
    """Inverse of the transformation applied to the expression file."""
    if scale == "log2log2":
        return np.power(2.0, np.power(2.0, v) - 1.0) - 1.0
    if scale == "log2":
        return np.power(2.0, v) - 1.0
    if scale == "linear":
        return v
    raise ValueError(scale)

par_y = expr.index.str.endswith("_PAR_Y")
print(f"removing {par_y.sum()} *_PAR_Y rows (max value: {expr[par_y].to_numpy().max():.3g})")
expr = expr.loc[~par_y]

lin = pd.DataFrame(to_linear(expr.to_numpy(dtype=np.float64), INPUT_SCALE),
                   index=expr.index, columns=expr.columns)
lin[lin < 0] = 0.0
ens = lin.index.str.split(".").str[0]

GUARD = {"ENSG00000111640": "GAPDH", "ENSG00000075624": "ACTB"}
for e, name in GUARD.items():
    med = float(np.median(lin.loc[ens == e].to_numpy()))
    alt = float(np.median(to_linear(expr.loc[ens == e].to_numpy(dtype=np.float64), "log2")))
    print(f"{name}: median FPKM = {med:,.1f} with INPUT_SCALE='{INPUT_SCALE}'  (it would be {alt:,.1f} with 'log2')")
    assert med > 100, f"{name} median FPKM {med:.1f} < 100: INPUT_SCALE is wrong"

In [ ]:
def load_or_build_map(ens_ids, path, refresh=False):
    """Ensembl gene -> Entrez gene, one Entrez per Ensembl (best MyGene.info score)."""
    if path.exists() and not refresh:
        m = pd.read_csv(path, dtype=str)
        print(f"using cached map {path.name}: {len(m):,} Ensembl IDs")
        return m
    import mygene
    mg = mygene.MyGeneInfo()
    res = mg.querymany(sorted(set(ens_ids)), scopes="ensembl.gene", fields="entrezgene",
                       species="human", as_dataframe=True, verbose=False).reset_index()
    res = res.rename(columns={"query": "ensembl"})
    if "notfound" in res:
        res = res[res["notfound"].isna() | (res["notfound"] == False)]   # noqa: E712
    res = res.dropna(subset=["entrezgene"])
    res = res.sort_values("_score", ascending=False).drop_duplicates("ensembl")
    m = pd.DataFrame({"ensembl": res["ensembl"].astype(str),
                      "entrez": res["entrezgene"].astype(str).str.replace(r"\.0$", "", regex=True)})
    m["source"] = f"MyGene.info (mygene {mygene.__version__}), {datetime.date.today()}"
    m.to_csv(path, index=False)
    print(f"saved {path.name}: {len(m):,} Ensembl IDs with an Entrez ID")
    return m

emap = load_or_build_map(ens, MAP_FILE, REFRESH_MAP)
emap = dict(zip(emap["ensembl"], emap["entrez"]))
entrez = pd.Series(ens, index=lin.index).map(emap)
print(f"{entrez.notna().sum():,} of {len(entrez):,} Ensembl IDs have an Entrez ID")

lin_e = lin.loc[entrez.notna()]
lin_e.index = entrez[entrez.notna()].to_numpy()
n_dup = lin_e.index.duplicated(keep=False).sum()
agg = lin_e.groupby(level=0).max() if DUPLICATE_RULE == "max" else lin_e.groupby(level=0).mean()
agg = agg.sort_index(key=lambda s: s.astype(int))
print(f"{n_dup} rows share an Entrez ID -> {agg.shape[0]:,} unique Entrez genes ({DUPLICATE_RULE})")
assert agg.index.is_unique

In [ ]:
# write one file per sample; the folder contains only *_Xomics.txt
XOMICS_DIR.mkdir(parents=True, exist_ok=True)
manifest = []
for s in agg.columns:
    v = agg[s]
    if WRITE_XOMICS:
        pd.DataFrame({"genes": agg.index, "expVal": v.to_numpy()}).to_csv(
            XOMICS_DIR / f"{s}_Xomics.txt", index=False, float_format="%.6g")
    manifest.append({"sample": s, "file": f"{s}_Xomics.txt", "n_genes": len(v),
                     "n_zero": int((v == 0).sum()), "n_below_1FPKM": int((v < 1).sum()),
                     "median_FPKM": float(v.median())})
manifest = pd.DataFrame(manifest)
manifest.to_csv(PROC_DIR / "xomics_manifest.csv", index=False)

stray = [f for f in os.listdir(XOMICS_DIR) if f.endswith(".txt") and not f.endswith("_Xomics.txt")]
assert not stray, f"non-Xomics .txt files in {XOMICS_DIR} (modelGen.m would read them): {stray[:5]}"
print(f"{len(manifest)} files in {XOMICS_DIR}" if WRITE_XOMICS else "WRITE_XOMICS = False: no files written")
manifest.describe().round(1)

**Checks** on the written values: pseudo-autosomal genes are no longer halved, and how many Recon3D genes each sample marks as inactive.

In [ ]:
# PAR genes: value must equal the chrX copy (not half of it)
for e, name in {"ENSG00000002586": "CD99", "ENSG00000169100": "SLC25A6"}.items():
    sym = emap.get(e)
    if sym is not None and sym in agg.index:
        x = lin.loc[ens == e].max().iloc[:3].round(2).tolist()
        print(f"{name} (Entrez {sym}): written {agg.loc[sym].iloc[:3].round(2).tolist()}  chrX copy {x}")

if RECON3D_GENES.exists():
    rg = pd.read_excel(RECON3D_GENES).iloc[:, 0].astype(str).str.split(".").str[0].unique()
    present = agg.index.intersection(rg)
    sub = agg.loc[present]
    print(f"\nRecon3D genes: {len(rg):,}; present in the files: {len(present):,} "
          f"({len(present)/len(rg):.1%}); absent (NaN in XomicsToModel): {len(rg)-len(present):,}")
    q = pd.DataFrame({"zero": (sub == 0).sum(), "inactive_below_1FPKM": (sub < 1).sum(),
                      "active": (sub >= 1).sum()})
    print("Recon3D genes per sample (median [min-max]):")
    for c in q:
        print(f"  {c:22s} {q[c].median():6.0f} [{q[c].min()}-{q[c].max()}]")
else:
    print(f"(optional check skipped: {RECON3D_GENES} not found)")

## 4. Processed clinical table

One row per expression sample. Sources: GDC clinical (stage, TNM, histology, treatment, demographics, tissue source site), Xena MetaData (ER/PR/HER2, PAM50, menopausal status, genetic ancestry; per patient) and Xena survival (OS, per sample).
Harmonisation: stage → I–IV, T → T1–T4, N → N0–N3, M → M0/M1 (X and missing → NaN); histology → IDC / ILC / Mixed / Other; receptor subtype → HR+/HER2−, HER2+, TNBC (indeterminate/equivocal → NaN); age as integer years.

In [ ]:
def first_match(x, pattern):
    m = re.match(pattern, str(x))
    return m.group(1) if m else np.nan

def collapse_stage(x):
    m = re.match(r"Stage (IV|III|II|I)", str(x))
    return m.group(1) if m else np.nan

def collapse_histology(x):
    return {"Infiltrating duct carcinoma, NOS": "IDC",
            "Lobular carcinoma, NOS": "ILC",
            "Infiltrating duct and lobular carcinoma": "Mixed IDC/ILC"}.get(x, np.nan if pd.isna(x) else "Other")

def parse_treatments(types, flags):
    """GDC stores parallel lists: ['Radiation Therapy, NOS', 'Pharmaceutical Therapy, NOS'] / ['yes','no']."""
    out = {"radiotherapy": np.nan, "pharmaceutical_therapy": np.nan}
    try:
        t, f = eval(types), eval(flags)          # strings of python lists written by the GDC export
    except Exception:
        return pd.Series(out)
    for ti, fi in zip(t, f):
        key = "radiotherapy" if "Radiation" in ti else "pharmaceutical_therapy" if "Pharmaceutical" in ti else None
        if key and fi in ("yes", "no"):
            out[key] = fi == "yes"
    return pd.Series(out)

c = clin.set_index("sample")
cp = pd.DataFrame(index=inv["sample"])
cp["patient"]      = inv.set_index("sample")["patient"]
cp["sample_type"]  = inv.set_index("sample")["sample_type"]
cp["tss_code"]     = inv.set_index("sample")["tss_code"]
cp["tss_name"]     = inv.set_index("sample")["tss_name"]
cp["age_years"]    = c["age_at_index.demographic"].reindex(cp.index).round().astype("Int64")
cp["sex"]          = c["gender.demographic"].reindex(cp.index)
cp["race"]         = c["race.demographic"].reindex(cp.index).replace({"not reported": np.nan})
cp["ethnicity"]    = c["ethnicity.demographic"].reindex(cp.index).replace({"not reported": np.nan})
cp["vital_status"] = c["vital_status.demographic"].reindex(cp.index)
cp["stage"]        = c["ajcc_pathologic_stage.diagnoses"].reindex(cp.index).map(collapse_stage)
cp["T"]            = c["ajcc_pathologic_t.diagnoses"].reindex(cp.index).map(lambda x: first_match(x, r"(T[1-4])"))
cp["N"]            = c["ajcc_pathologic_n.diagnoses"].reindex(cp.index).map(lambda x: first_match(x, r"(N[0-3])"))
cp["M"]            = c["ajcc_pathologic_m.diagnoses"].reindex(cp.index).map(lambda x: first_match(x, r"c?(M[01])"))
cp["histology"]    = c["primary_diagnosis.diagnoses"].reindex(cp.index).map(collapse_histology)
cp["primary_diagnosis"] = c["primary_diagnosis.diagnoses"].reindex(cp.index)
cp["prior_treatment"]   = c["prior_treatment.diagnoses"].reindex(cp.index).map({"Yes": True, "No": False})
tr = c.reindex(cp.index).apply(lambda r: parse_treatments(r["treatment_type.treatments.diagnoses"],
                                                          r["treatment_or_therapy.treatments.diagnoses"]), axis=1)
cp = cp.join(tr)

# MetaData (per patient)
m = meta.rename(columns={"Patient ID": "patient"}).drop_duplicates("patient").set_index("patient")
low = lambda s: s.astype(str).str.strip().str.lower().replace({"nan": np.nan})
for src, dst in [("ER", "ER"), ("PR", "PR"), ("HER2", "HER2"), ("Subtype", "PAM50"),
                 ("Menopausal Status", "menopausal_status"), ("Genetic Ancestry", "genetic_ancestry"),
                 ("Cancer Type", "cancer_type_metadata")]:
    cp[dst] = cp["patient"].map(m[src])
for r in ["ER", "PR", "HER2"]:
    cp[r] = low(cp[r]).where(low(cp[r]).isin(["positive", "negative"]))
hr_pos = (cp.ER == "positive") | (cp.PR == "positive")
hr_neg = (cp.ER == "negative") & (cp.PR == "negative")
cp["receptor_subtype"] = np.select(
    [cp.HER2 == "positive", hr_pos & (cp.HER2 == "negative"), hr_neg & (cp.HER2 == "negative")],
    ["HER2+", "HR+/HER2-", "TNBC"], default=None)
cp["receptor_subtype"] = cp["receptor_subtype"].replace({None: np.nan})
cp["in_metadata"] = inv.set_index("sample")["in_metadata"]

# survival (per sample; fall back to the patient's tumour sample for normals)
s = surv.drop_duplicates("sample").set_index("sample")
s_pat = surv.drop_duplicates("_PATIENT").set_index("_PATIENT")
cp["OS_days"]  = s["OS.time"].reindex(cp.index).fillna(cp["patient"].map(s_pat["OS.time"]))
cp["OS_event"] = s["OS"].reindex(cp.index).fillna(cp["patient"].map(s_pat["OS"])).astype("Int64")

cp = cp.reset_index()
cp.to_csv(PROC_DIR / "clinical_processed.csv", index=False)
inv.to_csv(PROC_DIR / "sample_inventory.csv", index=False)
print(f"clinical_processed.csv: {cp.shape}")

In [ ]:
# completeness of each variable among tumour samples
tum_cp = cp[cp.sample_type == "Tumor"]
compl = (tum_cp.drop(columns=["sample", "patient"]).notna().mean() * 100).round(1).sort_values()
print("Completeness among tumour samples (%):")
print(compl.to_string())
print("\nReceptor subtype vs PAM50 (tumours):")
pd.crosstab(tum_cp.receptor_subtype.fillna("NA"), tum_cp.PAM50.fillna("NA"))

## 5. Anonymised clinical table (supplementary material)

The editor rejected the previous supplementary files for containing identifiable data. This table replaces TCGA barcodes by sequential codes, contains no dates, gives age as an integer (top-coded at 89, i.e. "89" means 89 or older) and survival time in whole months. The key linking codes to barcodes is saved separately and **must not** be shared.

In [ ]:
anon = cp.copy()
rng = np.random.default_rng(20261005)                      # codes do not follow barcode order
pats = rng.permutation(anon["patient"].unique())
pat_code = {p: f"P{i+1:04d}" for i, p in enumerate(pats)}
anon.insert(0, "patient_code", anon["patient"].map(pat_code))
anon.insert(0, "sample_code", anon["patient_code"] + "_" + anon["sample_type"].str[0] +
            anon.groupby(["patient", "sample_type"]).cumcount().add(1).astype(str))
anon = anon.sort_values("sample_code").reset_index(drop=True)
key = anon[["sample_code", "patient_code", "sample", "patient"]]
key.to_csv(PROC_DIR / "anonymization_key_DO_NOT_SHARE.csv", index=False)

keep = ["sample_code", "patient_code", "sample_type", "age_years", "sex", "race", "ethnicity",
        "stage", "T", "N", "M", "histology", "ER", "PR", "HER2", "receptor_subtype", "PAM50",
        "menopausal_status", "genetic_ancestry", "radiotherapy", "pharmaceutical_therapy",
        "OS_months", "OS_event"]
anon["age_years"] = anon["age_years"].clip(upper=89)
anon["OS_months"] = (anon["OS_days"] / 30.44).round().astype("Int64")
out = anon[keep]
assert not out.astype(str).apply(lambda col: col.str.contains("TCGA")).any().any()
out.to_csv(SUPPL_DIR / "S_Table_clinical_anonymized.csv", index=False)
print(f"{SUPPL_DIR / 'S_Table_clinical_anonymized.csv'}: {out.shape}")
out.head()

## 6. Run log

In [ ]:
log = {
    "notebook": "BRCA_Metabolic_Flux_Pipeline_01.ipynb",
    "run_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
    "platform": platform.platform(),
    "inputs": {p.name: os.path.getsize(p) for p in [EXPRESSION_FILE, CLINICAL_FILE, SURVIVAL_FILE, METADATA_FILE]},
    "parameters": {"INPUT_SCALE": INPUT_SCALE, "DUPLICATE_RULE": DUPLICATE_RULE, "SAMPLE_LIMIT": SAMPLE_LIMIT,
                   "WRITE_XOMICS": WRITE_XOMICS, "XOMICS_THRESHOLD": XOMICS_THRESHOLD},
    "n_samples": int(len(inv)), "n_patients": int(inv.patient.nunique()),
    "n_entrez_genes": int(agg.shape[0]),
    "sample_types": inv.sample_type.value_counts().to_dict(),
}
json.dump(log, open(PROC_DIR / "run_log_01.json", "w"), indent=1)
print(json.dumps(log, indent=1))
print("\nmodelGen.m: point the transcriptomics folder to", XOMICS_DIR,
      f"\n            and set param.transcriptomicThreshold = {XOMICS_THRESHOLD};")